This notebook uses a LLM to answer questions 

In [1]:
import platform
import requests

# For the paper analyser
import torch
import transformers
import argparse
import logging
import json
import os
import accelerate

c:\Users\thiba\anaconda3\envs\evidence_db2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#  Get ChromaDB Collection

In [2]:
!pip install chromadb
#!pip install sentence_transformers

In [3]:
#get chromaDB and collection (collection must have been created and populated previously)
import chromadb
from chromadb.utils import embedding_functions

CHROMA_DATA_PATH = "chroma_data2/"
COLLECTION_NAME = "searchable_db_collection"

client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)
collection = client.get_collection(name="searchable_db_collection")

### USE LLama to answer a query

In [4]:
from openai import OpenAI

client = OpenAI(
        api_key = "W2oF2Q2NLaTmqj7LGOiJwp9Pdi47Rhhn",
        base_url="https://api.deepinfra.com/v1/openai",
    )

SYSTEM_MSG  = "You are a helpful systematic reviewing assistant"

def generateFromPrompt(promptStr,maxTokens=100):
    messages=[
    {"role": "system", "content": SYSTEM_MSG},
    {"role": "user", "content": promptStr}
    ]
    completion = client.chat.completions.create(
    model="meta-llama/Meta-Llama-3.1-70B-Instruct",
    messages=messages)
    response=completion.choices[0].message.content
    return(response)

In [5]:
prompt = "Please answer the following question using the following paper titles and abstracts."
query = "What are the expected outcomes for a middle-aged man with prostate cancer stage III? What are possible treatments?"
NB_PAPERS_LLM = 3


query_results = collection.query(
    query_texts=[query],
    n_results=NB_PAPERS_LLM,
)

title_and_abst = ",".join(query_results["documents"][0])

answer = generateFromPrompt(prompt + query + title_and_abst)

print("Retrieved from ",query_results["ids"][0], \
      "\n Titles: \n", {query_results['metadatas'][0][i]['titles'] for i in range(NB_PAPERS_LLM)}, \
      "\n \n Answer:",answer,\
      "\n\nTitle and abstract:",query_results['documents'][0])

Retrieved from  ['5741', '5557', '3119'] 
 Titles: 
 {'Retrospective Analysis of Clinico-Epidimological Factors in Prostatic Cancer', 'The Future of Advanced Prostate Cancer Treatment', 'A Comment on the International Society of Geriatric Oncology Guidelines: Evidence-Based Advice for the Clinical Setting'} 
 
 Answer: Based on the paper titles and abstracts, here are the expected outcomes and possible treatments for a middle-aged man with prostate cancer stage III:

Expected Outcomes:

* The median overall survival for patients with prostate cancer is around 31 months, as reported in the Retrospective Analysis of Clinico-Epidemiological Factors in Prostatic Cancer study.
* The median progression-free survival (PFS) is around 17 months, and the median disease-free survival (DFS) is around 29 months, as reported in the same study.

Possible Treatments:

* The treatment options for prostate cancer vary depending on age, stage, and grade of cancer, as well as other medical conditions, as 